In [8]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [9]:
# Cargar el subset que guardaste
df = pd.read_csv(r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-reducido-merge\df_subset_pa_ap.csv")
print(f"Total imágenes: {len(df)}")

# Las 7 clases con suficientes positivos
TARGET_CLASSES = ['No Finding', 'Cardiomegaly', 'Pleural Effusion', 
                  'Atelectasis', 'Consolidation', 'Pneumonia', 'Support Devices']

# U-zeros: -1.0 (incierto) y NaN → 0; solo 1.0 cuenta como positivo
for c in TARGET_CLASSES:
    df[c] = df[c].fillna(0).replace(-1.0, 0).astype(int)

# Verificar
print("\nDistribución final (positivos por clase):")
print(df[TARGET_CLASSES].sum())

df['full_path'] = df['full_path'].str.replace(
    'subset_pa_ap',
    'dataset_limpieza_texto',
    regex=False
)


missing = df[~df['full_path'].apply(os.path.exists)]
print(f"Rutas faltantes: {len(missing)} de {len(df)}")

Total imágenes: 315

Distribución final (positivos por clase):
No Finding          215
Cardiomegaly         25
Pleural Effusion     27
Atelectasis          31
Consolidation         4
Pneumonia             9
Support Devices      48
dtype: int64
Rutas faltantes: 0 de 315


In [10]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, val_idx = next(splitter.split(df, groups=df['subject_id']))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val   = df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(df_train)}  ({df_train['subject_id'].nunique()} pacientes)")
print(f"Val:   {len(df_val)}  ({df_val['subject_id'].nunique()} pacientes)")

# Verificar que no hay solapamiento de pacientes
overlap = set(df_train['subject_id']) & set(df_val['subject_id'])
print(f"Pacientes solapados (debe ser 0): {len(overlap)}")

Train: 252  (226 pacientes)
Val:   63  (57 pacientes)
Pacientes solapados (debe ser 0): 0


In [11]:
# Normalización ImageNet (DenseNet-121 viene preentrenada en ImageNet)
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CXRDataset(Dataset):
    def __init__(self, df, classes, transform):
        self.df = df.reset_index(drop=True)
        self.classes = classes
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['full_path']).convert('RGB')
        img = self.transform(img)
        labels = torch.tensor(row[self.classes].values.astype(np.float32))
        return img, labels

train_ds = CXRDataset(df_train, TARGET_CLASSES, transform_train)
val_ds   = CXRDataset(df_val,   TARGET_CLASSES, transform_val)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Test rápido: una imagen
x, y = next(iter(train_loader))
print(f"Batch shape: {x.shape}, labels shape: {y.shape}")

Batch shape: torch.Size([16, 3, 224, 224]), labels shape: torch.Size([16, 7])


In [12]:
def build_model(num_classes):
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    # Reemplazar la cabeza final para multi-label
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)
    return model

model = build_model(num_classes=len(TARGET_CLASSES)).to(device)

# Loss multi-label: BCE con logits (numéricamente estable)
criterion = nn.BCEWithLogitsLoss()

# Optimizador
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(f"Modelo: DenseNet-121, {sum(p.numel() for p in model.parameters()):,} parámetros")

Modelo: DenseNet-121, 6,961,031 parámetros


In [13]:
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x).cpu().numpy()
            all_logits.append(logits)
            all_labels.append(y.numpy())
    logits = np.vstack(all_logits)
    labels = np.vstack(all_labels)
    
    aucs = {}
    for i, c in enumerate(TARGET_CLASSES):
        if labels[:, i].sum() > 0 and labels[:, i].sum() < len(labels):
            aucs[c] = roc_auc_score(labels[:, i], logits[:, i])
        else:
            aucs[c] = np.nan
    return aucs

NUM_EPOCHS = 10
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
        pbar.set_postfix(loss=loss.item())
    
    epoch_loss /= len(train_ds)
    aucs = evaluate(model, val_loader)
    mean_auc = np.nanmean(list(aucs.values()))
    
    print(f"Epoch {epoch} | Loss: {epoch_loss:.4f} | Mean AUC (val): {mean_auc:.4f}")
    history.append({"epoch": epoch, "loss": epoch_loss, "mean_auc": mean_auc, **aucs})

history_df = pd.DataFrame(history)
print("\nHistorial completo:")
print(history_df)

Epoch 1/10: 100%|██████████| 16/16 [00:21<00:00,  1.37s/it, loss=0.485]


Epoch 1 | Loss: 0.5705 | Mean AUC (val): 0.8240


Epoch 2/10: 100%|██████████| 16/16 [00:21<00:00,  1.31s/it, loss=0.346]


Epoch 2 | Loss: 0.3656 | Mean AUC (val): 0.8949


Epoch 3/10: 100%|██████████| 16/16 [00:18<00:00,  1.13s/it, loss=0.26] 


...

Epoch 8 | Loss: 0.1150 | Mean AUC (val): 0.8558


Epoch 9/10: 100%|██████████| 16/16 [00:22<00:00,  1.42s/it, loss=0.0597]


Epoch 9 | Loss: 0.1040 | Mean AUC (val): 0.8329


Epoch 10/10: 100%|██████████| 16/16 [00:22<00:00,  1.38s/it, loss=0.0781]


Epoch 10 | Loss: 0.0909 | Mean AUC (val): 0.8121

Historial completo:
   epoch      loss  mean_auc  No Finding  Cardiomegaly  Pleural Effusion  \
0      1  0.570536  0.823996    0.837209      0.751724          0.845029   
1      2  0.365628  0.894886    0.846512      0.934483          0.862573   
2      3  0.271531  0.893502    0.848837      0.931034          0.868421   
3      4  0.222789  0.912911    0.860465      0.962069          0.906433   
4      5  0.182888  0.873543    0.902326      0.900000          0.783626   
5      6  0.151372  0.866687    0.882558      0.831034          0.845029   
6      7  0.131394  0.918300    0.901163      0.913793          0.900585   
7      8  0.115028  0.855770    0.877907      0.868966          0.774854   
8      9  0.103993  0.832873    0.872093      0.768966          0.733918   
9     10  0.090921  0.812102    0.846512      0.696552          0.792398   

   Atelectasis  Consolidation  Pneumonia  Support Devices  
0     0.841564            NaN   0

In [14]:
print("=" * 60)
print("BASELINE — DenseNet-121 sin limpieza")
print("=" * 60)
print(f"Train: {len(df_train)} imgs | Val: {len(df_val)} imgs")
print(f"Mejor mean AUC: {history_df['mean_auc'].max():.4f} (epoch {history_df['mean_auc'].idxmax() + 1})")
print("\nAUC por clase (última epoch):")
for c in TARGET_CLASSES:
    print(f"  {c:<25} {history_df.iloc[-1][c]:.4f}")

# Guardar para comparación futura con el dataset limpio
history_df.to_csv(r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\baseline_history_wtext.csv", index=False)
print("\n✓ Historial guardado en baseline_history_raw.csv")

BASELINE — DenseNet-121 sin limpieza
Train: 252 imgs | Val: 63 imgs
Mejor mean AUC: 0.9183 (epoch 7)

AUC por clase (última epoch):
  No Finding                0.8465
  Cardiomegaly              0.6966
  Pleural Effusion          0.7924
  Atelectasis               0.8909
  Consolidation             nan
  Pneumonia                 0.8197
  Support Devices           0.8265

✓ Historial guardado en baseline_history_raw.csv


In [15]:
import torch
print(torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Versión CUDA:", torch.version.cuda)

2.13.0.dev20260513+cu132
CUDA disponible: True
Versión CUDA: 13.2
